
## Grafico top 10 clientes em risco
--------------------------------------
Gera um gráfico de barras com os 10 clientes ativos com MAIOR probabilidade
de churn, a partir da tabela já gerada pelo script
"gerar_clientes_propensos_ao_churn.py".

PRÉ-REQUISITO:
  Rodar antes o script "gerar_clientes_propensos_ao_churn.py", que gera o
  arquivo "clientes_propensos_ao_churn.csv" usado aqui.

COMO RODAR:
    pip install pandas matplotlib
    python grafico_top10_clientes_risco.py

SAÍDA:
  - top10_clientes_risco.png

In [ ]:
import os
import sys
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURAÇÕES
# ============================================================
CAMINHO_CSV = 'clientes_propensos_ao_churn.csv'

# Define a pasta de saída e garante a criação dela de forma segura
PASTA_SAIDA = 'outputs'
os.makedirs(PASTA_SAIDA, exist_ok=True)

# Define o caminho final combinando a pasta com o nome do arquivo
SAIDA_GRAFICO = os.path.join(PASTA_SAIDA, 'top10_clientes_risco.png')

CORES_RISCO = {'Alto': '#E74C3C', 'Médio': '#F39C12', 'Baixo': '#27AE60'}


def carregar_top10(caminho):
    if not os.path.isfile(caminho):
        raise FileNotFoundError(
            f"\nArquivo não encontrado: '{caminho}'\n"
            f"Pasta atual: {os.getcwd()}\n"
            f"Dica: rode primeiro o script 'gerar_clientes_propensos_ao_churn.py' "
            f"para gerar esse arquivo, e execute os dois na mesma pasta."
        )
    df = pd.read_csv(caminho)
    return df.sort_values('Ranking').head(10)


def gerar_grafico(top10, caminho_saida):
    top10 = top10.iloc[::-1]  # inverte p/ o Top 1 aparecer no topo do gráfico
    cores = [CORES_RISCO.get(f, '#94A3B8') for f in top10['Faixa_Risco']]

    fig, ax = plt.subplots(figsize=(9, 6))
    barras = ax.barh(top10['ID_Cliente'], top10['Probabilidade_Churn'], color=cores)

    for barra, prob in zip(barras, top10['Probabilidade_Churn']):
        ax.text(prob + 1, barra.get_y() + barra.get_height() / 2,
                f'{prob:.1f}%', va='center', fontsize=10, fontweight='bold')

    ax.set_xlabel('Probabilidade de Churn (%)')
    ax.set_title('Top 10 Clientes Ativos com Maior Risco de Churn', fontsize=13, fontweight='bold')
    ax.set_xlim(0, max(top10['Probabilidade_Churn']) + 12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Legenda de faixas de risco (abaixo do gráfico, para não sobrepor as barras)
    handles = [plt.Rectangle((0, 0), 1, 1, color=cor) for cor in CORES_RISCO.values()]
    ax.legend(handles, CORES_RISCO.keys(), title='Faixa de Risco',
              loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)

    plt.tight_layout()
    plt.savefig(caminho_saida, dpi=130)
    plt.close()


def main():
    print("1/2 — Carregando os 10 clientes de maior risco...")
    try:
        top10 = carregar_top10(CAMINHO_CSV)
        print(top10[['Ranking', 'ID_Cliente', 'Probabilidade_Churn', 'Faixa_Risco']].to_string(index=False))

        print("\n2/2 — Gerando gráfico...")
        gerar_grafico(top10, SAIDA_GRAFICO)

        # Mostra o caminho absoluto para facilitar a localização do arquivo
        caminho_absoluto = os.path.abspath(SAIDA_GRAFICO)
        print(f"\nConcluído! Gráfico salvo em: {caminho_absoluto}")
        
    except FileNotFoundError as e:
        print(e)


if __name__ == '__main__':
    main()
